# The fault as a temperature — the paper figure

For each (model, eval fault) the logit-sampling pipeline gives the fault-marginalised
predictive $\bar p = \mathbb{E}_{\text{chip}}[\operatorname{softmax}(z)]$ over a fixed
context set. We fit the *single* global temperature that best explains it in forward KL,
$$T^\star = \arg\min_T \sum_c \mathrm{KL}\!\left(\bar p_c \,\|\, \operatorname{softmax}(z_{0,c}/T)\right),$$
and split the total distortion exactly ($T^\star$ is the M-projection onto the joint
exponential family, so this is a Pythagorean identity):
$$\underbrace{\mathrm{KL}(\bar p\|\text{clean})}_{\text{total}}
= \underbrace{\mathrm{KL}(\bar p\|p_{T^\star})}_{\text{residual — not a temperature}}
+ \underbrace{\mathrm{KL}(p_{T^\star}\|\text{clean})}_{\text{explained by the temperature}} .$$

Top panel: $T^\star$ against the eval fault. Bottom: the residual as a fraction of the model's
own clean loss $L_0$ — the part no single temperature can account for. Colour carries training
duration (light → dark = more tokens per parameter); blue is the cleanly-trained arm, orange
the fault-trained one.


In [12]:
import os, sys

# Repo root on the path (for `nano_llama` and `scaling_law`), from either cwd.
_HERE = os.getcwd()
for _cand in (_HERE, os.path.dirname(_HERE)):
    if os.path.isdir(os.path.join(_cand, "nano_llama")) and _cand not in sys.path:
        sys.path.insert(0, _cand)

import numpy as np
import matplotlib.pyplot as plt
plt.style.use('default')

from experiment_util import process_logit_marginals as plm

# The DIGEST, not the raw array: process_logit_marginals.py reduces the ~566 GB campaign
# once to ~10 MB -- the per-(run, fault) scalars these figures draw, plus a subsampled
# scatter cloud per panel of the 2x2 below. Re-run it when the array grows.
DIGEST_DIR = "/mnt/storage/logit_digests/d512_tpp"   # <-- SET

dg = plm.load_digest(DIGEST_DIR)
runs = dg.runs                               # DigestRun: same attribute names the figures always used
FIT = dg.fit                                 # (job, p_eval) -> the fitted scalars
L0 = {r.job: r.clean_loss for r in runs}     # each run's clean cross-entropy (nats)

# The digest is built from whichever jobs had finished, so a run can carry a short fault
# grid and draw a curve that stops early. Say so rather than plotting it silently.
_full = max(len(r.p_eval) for r in runs)
_short = [r for r in runs if len(r.p_eval) < _full]
if _short:
    print(f"!! {len(_short)} of {len(runs)} run(s) are SHORT of the full {_full}-point fault grid "
          f"(fewest: {min(len(r.p_eval) for r in _short)}). They stop early in every panel; re-run "
          f"process_logit_marginals.py with REQUIRE_DONE = True to drop them.\n")

arms = sorted({r.p_train for r in runs})
tpps = sorted({round(r.tokens_per_param) for r in runs})
shapes = sorted({r.size_key for r in runs})
r0 = runs[0]
print(f"{len(runs)} run(s), {len(FIT)} (model, p_eval) point(s) | C={r0.n_contexts} contexts, "
      f"V={r0.vocab_size}, M={r0.n_chips} chips | digest {DIGEST_DIR}")
print(f"architecture(s): {', '.join(shapes)} | arms p_train = {', '.join(f'{a:g}' for a in arms)} | "
      f"tokens/param: {', '.join(f'{t:g}' for t in tpps)}\n")

# ================= seeds =================
# The sweep trains seed replicates of every (duration, p_train) cell, and the digest keeps
# each as its own run -- it is an inventory of what ran, and an average baked in there could
# not be undone without another full pass. Aggregation belongs here, as a presentation choice.
#
# MEDIAN, not mean, and taken on the PLOTTED quantity. Both panels draw a monotone transform of a
# per-run scalar (T-1 on a log axis; capacity_retained, which is monotone decreasing in the residual),
# and the median COMMUTES with a monotone transform -- so "aggregate then transform" and "transform
# then aggregate" agree exactly, and the question of which is right does not arise. Measured on this
# array: median differs by <=0.06% between the two orders (pure even-n midpoint averaging), where the
# MEAN carries a Jensen gap up to 0.57% that grows with the fault. The band is the seed
# inter-quartile range, so what is drawn is the actual scatter of the thing on the axis.
#
# The spread is not cosmetic: across 8 seeds of one cell, T varies ~0.5% but the residual varies
# 8-27%, so a single-seed curve would be reporting one draw as if it were the effect.
CELL = {}                                    # (p_train, round(tok/param)) -> [runs], the seed group
for r in runs:
    CELL.setdefault((r.p_train, round(r.tokens_per_param)), []).append(r)

def seed_band(cell_key, p_eval, value, lo=25, hi=75):
    """(median, lo, hi, n_seeds) of `value(job, p_eval)` across a cell's seed replicates.

    `value` is evaluated PER SEED and then reduced, so the band is the spread of the plotted quantity.
    Seeds missing this p_eval are skipped (a cell still filling can be ragged), and n_seeds says how
    many actually contributed -- a median over 3 seeds and one over 8 are not the same object.
    """
    v = np.array([value(r.job, p_eval) for r in CELL[cell_key] if (r.job, p_eval) in FIT], float)
    v = v[np.isfinite(v)]
    if v.size == 0:
        return np.nan, np.nan, np.nan, 0
    return float(np.median(v)), float(np.percentile(v, lo)), float(np.percentile(v, hi)), int(v.size)

def cell_grid(cell_key):
    """The eval faults every seed in this cell has -- the grid a banded curve can be drawn over."""
    return sorted(set.intersection(*(set(r.p_eval) for r in CELL[cell_key])))

_n_seeds = {k: len(v) for k, v in CELL.items()}
_full = max(_n_seeds.values())
_short = {k: n for k, n in _n_seeds.items() if n < _full}
print(f"{len(CELL)} (p_train, tok/param) cell(s), {_full} seed replicates each at full coverage")
if _short:
    print(f"!! {len(_short)} cell(s) have FEWER seeds -- their median is over less data:")
    for (p, t), n in sorted(_short.items()):
        print(f"     p_train={p:<6g} tok/param={t:<6g} {n}/{_full} seeds")
print()

# Per-cell summary, at three representative faults. Per-RUN rows would be one line per (run, p_eval) --
# tens of thousands of them -- which is an inventory, not a table you read.
hdr = (f"{'tok/param':>10}{'p_train':>9}{'seeds':>6}{'L_0':>7}{'p_eval':>9}"
       f"{'T* (IQR)':>22}{'explained':>11}{'resid/L0 (IQR)':>26}")
print(hdr); print("-" * len(hdr))
for (arm, tpp) in sorted(CELL, key=lambda k: (k[1], k[0])):
    g = cell_grid((arm, tpp))
    _L0 = float(np.median([L0[r.job] for r in CELL[(arm, tpp)]]))
    for pe in (g[0], g[len(g) // 2], g[-1]):
        T, Tlo, Thi, n = seed_band((arm, tpp), pe, lambda j, q: FIT[(j, q)]["T"])
        R, Rlo, Rhi, _ = seed_band((arm, tpp), pe,
                                   lambda j, q: FIT[(j, q)]["residual_mean"] / L0[j])
        fr, *_ = seed_band((arm, tpp), pe, lambda j, q: FIT[(j, q)]["frac"])
        print(f"{tpp:>10g}{arm:>9g}{n:>6}{_L0:>7.3f}{pe:>9.3g}"
              f"{f'{T:.3f} [{Tlo:.3f}, {Thi:.3f}]':>22}{fr:>11.1%}"
              f"{f'{R:.4f} [{Rlo:.4f}, {Rhi:.4f}]':>26}")

# ---- the clean scaling law, for pricing nats of distortion in parameters ------------------------
# The bottom panel reports the residual as CAPACITY: how far a model could shrink before its loss
# rose by the same number of nats. Each ARM is priced on its OWN law -- the clean arm on the
# p_train = 0 cohort, a fault-trained arm on the cohort fit at its own p_train -- so the number answers
# "what would it cost THIS kind of model to buy the distortion back by scaling up?". Point
# FIT_ARTIFACT at what fit_matched_scaling_law.py wrote.
#
# The cost of that choice, worth stating in the text: a matched-condition cohort describes models
# trained AND evaluated at its p, so away from p_eval = p_train it is an extrapolation, and the two
# arms are no longer in a common currency (pricing the faulted arm on the clean law instead moves its
# retained capacity by 1.2-1.4x -- small against the ~60x gap between arms, but not zero). The clean-law
# column in the endpoint table below is that sensitivity check.
from scaling_law import fit_store

FIT_ARTIFACT = "/media/trevor/data_flash/scaling_law_fits/2026-08-26-13-18-47"   # <-- SET
_an, _man = fit_store.load_analyses(FIT_ARTIFACT)
_need = sorted({r.p_train for r in runs})
_missing = [p for p in _need if p not in _an]
if _missing:
    raise KeyError(f"{FIT_ARTIFACT} has no cohort at p_train {_missing}, and each arm is priced on its "
                   f"own law. Fit an artifact that covers these arms, or price everything on the clean "
                   f"law instead (cohorts present: {sorted(_an)})")
LAWS = {p: _an[p]["fit"] for p in _need}      # arm p_train -> its matched-condition law
LAW = LAWS[min(_need)]                        # the reference law: clean if the array has a clean arm
# The law is always fit against TOTAL params.
def law_N(r):
    """This run's parameter count, in whichever convention the law used."""
    return float(r.n_params)

print(f"\nscaling laws from {FIT_ARTIFACT}   (N = total params, one per arm)")
# Reported in the CENTRED basis (scaling_law.surface): `amp_N` is the N-term's value AT the pivot N0
# -- not the old `A`, which was that term extrapolated to a one-parameter model -- and `alpha1` is its
# decay rate AT the pivot. With `alpha2` non-zero the effective exponent is alpha1 + 2*alpha2*log(N/N0)
# and varies along the ladder, so the 1/alpha1 column below is a slope read at N0, not "the" exponent.
_h = (f"{'arm p_train':>12}{'N-term@N0':>11}{'alpha1':>9}{'alpha2':>10}{'1/alpha1':>10}"
      f"{'10% of N-term costs':>21}")
print(_h); print("-" * len(_h))
for p, law in LAWS.items():
    print(f"{p:>12g}{law.amp_N:>11.3f}{law.alpha1:>9.4f}{law.alpha2:>10.4f}{1 / law.alpha1:>10.1f}"
          f"{100 * (1 - 1.1 ** (-1 / law.alpha1)):>20.0f}%")
if any(law.alpha2 for law in LAWS.values()):
    print(f"  (pivot N0 = {LAW.N0:.3g}; arms with alpha2 != 0 have a curved N-term -- "
          f"capacity_retained below inverts it exactly, so the curvature is not dropped)")


!! 64 of 384 run(s) are SHORT of the full 81-point fault grid (fewest: 80). They stop early in every panel; re-run process_logit_marginals.py with REQUIRE_DONE = True to drop them.

384 run(s), 31040 (model, p_eval) point(s) | C=256 contexts, V=8192, M=1000 chips | digest /mnt/storage/logit_digests/d512_tpp
architecture(s): d0512_L13 | arms p_train = 0, 0.02, 0.04, 0.06, 0.08, 0.12 | tokens/param: 5, 10, 20, 40, 80, 160, 320, 640

48 (p_train, tok/param) cell(s), 8 seed replicates each at full coverage

 tok/param  p_train seeds    L_0   p_eval              T* (IQR)  explained            resid/L0 (IQR)
----------------------------------------------------------------------------------------------------
         5        0     8  3.574    0.002  1.005 [1.005, 1.005]      12.2%   0.0001 [0.0001, 0.0001]
         5        0     8  3.574   0.0206  1.056 [1.055, 1.057]      24.3%   0.0079 [0.0070, 0.0081]
         5        0     8  3.574      0.2  1.792 [1.779, 1.797]      61.9%   0.2071 [0.

In [13]:
%matplotlib tk

# ---- paper figure: the global temperature, and what it leaves behind ---------------------
# Two panels sharing the x axis: T* on top, the non-temperature residual below. Hue carries
# the ARM (blue = cleanly trained, orange = fault-trained) and lightness the DURATION, which
# is what the comparison is about.
# ONE FIGURE PER FAULTED ARM: there is only one orange ramp, so overlaying two faulted arms
# draws the second in the first's colours.
FAULT_ARMS = [0.02, 0.04, 0.06, 0.08, 0.12]   # <-- SET: the NONZERO arms to make a figure for, one each
WITH_CLEAN = True                 # <-- SET: draw the clean arm as the reference in each figure
# NORMALIZED divides the fault axis by the figure's arm, so x = 1 is the condition that arm
# trained for. Self-contained per figure, but it puts the arms on DIFFERENT x scales. False
# plots raw p_eval instead, so the figures share one absolute axis and read side by side.
NORMALIZED = True                 # <-- SET: x = p_eval / p_train (True) or raw p_eval (False)
# None -> just draw them. Otherwise a TEMPLATE taking the arm, e.g.
#   "/home/trevor/figures/logit_temperature_p{arm:g}.pdf"
SAVE_PATH = None

import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D

# Ordinal ramps starting at ~250: the lighter steps fail the 2:1 contrast floor on white.
_RAMPS = {0.0: ["#86b6ef", "#3987e5", "#256abf", "#0d366b"],       # blue  -- clean arm
          "faulted": ["#f79a69", "#eb6834", "#c44a1e", "#8a3315"]}  # orange -- fault-trained arm
INK_SOFT = "#52514e"

_arms_have = sorted({r.p_train for r in runs})
_missing = [a for a in FAULT_ARMS if a not in _arms_have]
if _missing:
    raise KeyError(f"no runs at p_train {_missing}; this digest has {_arms_have}")
if any(a == 0 for a in FAULT_ARMS):
    raise ValueError("FAULT_ARMS holds the NONZERO arms -- the clean arm is the reference drawn in "
                     "every figure (WITH_CLEAN), not one of the figures")
if WITH_CLEAN and 0.0 not in _arms_have:
    raise KeyError(f"WITH_CLEAN is set but this digest has no p_train = 0 arm (has {_arms_have})")
if SAVE_PATH is not None and "{arm" not in SAVE_PATH:
    # There is one figure per arm now, so a fixed path would have each overwrite the last and leave
    # a single file that silently claims to be all of them.
    raise ValueError(f"SAVE_PATH must be a TEMPLATE containing {{arm}} now that there is one figure "
                     f"per faulted arm, e.g. '.../logit_temperature_p{{arm:g}}.pdf'; got {SAVE_PATH!r}")

tpp_list = sorted({round(r.tokens_per_param) for r in runs})
def _ramp(arm):
    steps = _RAMPS[0.0] if arm == 0.0 else _RAMPS["faulted"]
    cm = LinearSegmentedColormap.from_list(f"ord_{arm}", steps)
    return [cm(x) for x in np.linspace(0.0, 1.0, len(tpp_list))] if len(tpp_list) > 1 else [cm(0.75)]
COLOR = {a: dict(zip(tpp_list, _ramp(a))) for a in sorted({0.0, *FAULT_ARMS})}
def _col(r):
    return COLOR[r.p_train][round(r.tokens_per_param)]

def capacity_retained(delta_nats, N, law):
    """N_eff/N -- how far a model of size N could shrink, ALONG `law`, before its loss rose by
    `delta_nats`. An exchange rate, not a measurement: it prices distortion in parameters.

    `law` is the arm's own matched-condition fit, so a fault-trained model's distortion is priced in
    fault-trained parameters (what it would cost that model to scale the distortion away) rather than
    in clean ones. See the load cell for what that convention costs.

    Anchored on the LAW, not on the run's measured clean loss: n_term(N_eff) = n_term(N) + delta, so
    only the N-term enters and delta = 0 returns exactly 1. Anchoring on the measured L_0 instead
    would fold in the gap between the run and the fitted surface -- 0.16-0.27 nats on this array,
    which alone would read as 57-76% of the parameters gone at ZERO distortion. E and the D-term
    cancel (both are held fixed), so this is exact whatever fix_E the law was fit with.
    """
    # Solve n_term(N_eff) = n_term(N) + delta for N_eff, in the CENTRED basis:
    #     exp(a - alpha1*u - alpha2*u^2) = g,   u = log(N_eff/N0)
    # is a quadratic in u. Taken on the monotone branch in the cancellation-free form, so it stays
    # EXACT as alpha2 -> 0 (where it reduces to the old (1 + delta/LN)^(-1/alpha1)). Same inversion as
    # analysis/scaling_law_effective_capacity.ipynb -- the two must agree.
    N = np.asarray(N, float)
    g = law.n_term(N) + np.asarray(delta_nats, float)    # target N-term value, in nats
    with np.errstate(invalid="ignore", divide="ignore"):
        c = np.log(g) - law.a
        disc = law.alpha1 ** 2 - 4.0 * law.alpha2 * c
        n_eff = law.N0 * np.exp(-2.0 * c / (law.alpha1 + np.sqrt(disc)))
    return np.where((g > 0) & (disc >= 0), n_eff / N, np.nan)

def draw_arm(arm):
    """One figure: the clean reference against ONE fault-trained arm, plus its endpoint table.

    P_REF is `arm` itself, so the x axis reads p_eval / p_train with x = 1 the condition this arm
    was trained for -- true of every figure, which a single overlaid plot could not manage.
    """
    arm_list = ([0.0] if WITH_CLEAN else []) + [arm]
    P_REF = arm if NORMALIZED else 1.0        # 1.0 -> the axis is p_eval itself
    print(f"\n{'=' * 74}\np_train = {arm:g}"
          + ("  (vs the clean arm)" if WITH_CLEAN else "") + f"\n{'=' * 74}")
    # ONE curve per (arm, duration) CELL -- the median over its seed replicates, with the seed IQR as a
    # band. Drawing every seed as its own line would put 8 curves of identical colour on top of each
    # other, since hue carries the arm and lightness the duration; the band says the same thing and says
    # how wide it is. See the seed note in the load cell for why median (it commutes with both panels'
    # monotone transforms) and why this matters (the residual moves 8-27% across seeds).
    fig, (aT, aR) = plt.subplots(2, 1, figsize=(3.4, 3.4), sharex=True)
    _SHOWN = [k for k in sorted(CELL, key=lambda k: (k[0], k[1])) if k[0] in arm_list]
    for key in _SHOWN:
        arm, tpp = key
        r0c = CELL[key][0]                       # any seed: colour and N are cell properties, not seed ones
        xs = cell_grid(key)
        xn = np.array([q / P_REF for q in xs])
        col = COLOR[arm][tpp]

        tmid, tlo, thi = np.array([seed_band(key, q, lambda j, w: FIT[(j, w)]["T"] - 1.0)[:3]
                                   for q in xs]).T
        aT.fill_between(xn, tlo, thi, color=col, alpha=0.20, lw=0, zorder=2)
        aT.plot(xn, tmid, color=col, lw=1.5, solid_capstyle="round", zorder=3)

        cap = lambda j, w: float(capacity_retained(FIT[(j, w)]["residual_mean"], law_N(r0c), LAWS[arm]))
        rmid, rlo, rhi = np.array([seed_band(key, q, cap)[:3] for q in xs]).T
        aR.fill_between(xn, rlo, rhi, color=col, alpha=0.20, lw=0, zorder=2)
        aR.plot(xn, rmid, color=col, lw=1.5, solid_capstyle="round", zorder=3)

    # EXCESS temperature on a log axis, not T itself. The fault-trained arm lives in T = 1.005..1.53,
    # which on a linear T axis shared with the clean arm's 2.8 is a flat line against the T = 1 rule; the
    # excess T - 1 spreads that same arm over 0.005..0.53, so both arms are readable at once. It is also
    # the quantity with the natural zero -- "no flattening" is T - 1 = 0, which a log axis puts at the
    # bottom edge rather than needing a rule drawn across the panel.
    _dT = [seed_band(key, q, lambda j, w: FIT[(j, w)]["T"] - 1.0)[0]
           for key in _SHOWN for q in cell_grid(key)]
    if min(_dT) <= 0:
        raise ValueError("a fitted T* is <= 1 (the fault SHARPENED that predictive), which log(T-1) "
                         "cannot show; plot T on a linear axis for this array, or use symlog")
    aT.set_yscale("log")
    # Short labels: at 3.4 x 3.4 in the panels are squat, and the spelled-out forms
    # ("Residual KL(p_bar || p_T*) / L_0") are taller than the axes and collide with each other. The
    # definitions belong in the caption anyway.
    aT.set_ylabel(r"Excess temp.  ($T-1$)", fontsize=8)
    aR.set_yscale("log")
    aR.set_ylabel("Capacity retained" + "\n" + r"($\eta/N$, temp. excluded)", fontsize=8)
    aR.set_xscale("log")
    # The CONCRETE normaliser, not the symbol: each figure is one arm, so "p_eval / 0.04" tells the
    # reader what x = 1 actually is without going to the caption -- where "p_eval / p_train" would
    # leave them to remember which arm this panel is. Under NORMALIZED = False there is no divisor.
    aR.set_xlabel(rf"Inference-time fault-rate ($p_\mathrm{{eval}}/{arm:g}$)" if NORMALIZED
                  else r"Inference-time fault-rate ($p_\mathrm{eval}$)", fontsize=8)
    # Either range spans barely more than a decade or two, so the default log ticking labels a single
    # power of ten. Tick the round values that fall inside it, as plain numbers. The candidate list
    # covers both scales: the normalised axis lands in ~[0.02, 10], the raw one in ~[0.002, 0.2].
    _span_all = [q for key in _SHOWN for q in cell_grid(key)]
    _xt = [t for t in (0.001, 0.002, 0.003, 0.005, 0.01, 0.02, 0.03, 0.05,
                       0.1, 0.2, 0.3, 0.5, 1, 2, 3, 5, 10)
           if min(_span_all) / P_REF * 0.9 <= t <= max(_span_all) / P_REF * 1.1]
    # Exactly the data's span, no autoscale margin: every curve is drawn over the same fault grid, so the
    # padding was empty on both sides. sharex, so setting it on one panel sets both.
    _xspan = [q / P_REF for q in _span_all]
    aR.set_xlim(min(_xspan), max(_xspan))
    aR.set_xticks(_xt)
    aR.set_xticklabels([f"{t:g}" for t in _xt])
    aR.xaxis.set_minor_locator(mticker.NullLocator())
    for ax in (aT, aR):
        #ax.axvline(1.0, color="0.55", lw=0.7, zorder=2)      # run at the fault the faulted arm trained for
        # Plain numbers on the log y axes (1, 0.1, 0.01, ...) rather than powers of ten: both panels span
        # only a few decades of small ratios, which a reader takes in faster written out. Minor labels off
        # so a decade with room to spare cannot sprout 2, 3, 4...
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:g}"))
        ax.yaxis.set_minor_formatter(mticker.NullFormatter())
        ax.tick_params(labelsize=7.5, width=0.7, length=2.5, colors=INK_SOFT)
        ax.grid(True, which="major", axis="both", color="0.88", lw=0.5, zorder=0)
        ax.set_axisbelow(True)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
        for side in ("left", "bottom"):
            ax.spines[side].set_linewidth(0.7)

    # A two-entry key for the ARM only -- the duration ramp is left to the caption. Labels and colours
    # both come from the data (arm_list, the arm's own ramp at its middle step), so an array with different
    # training faults relabels itself. Lower right of the top panel: every curve rises left-to-right, so
    # that corner is the one place in either panel with nothing in it.
    _key = [Line2D([], [], color=COLOR[a][tpp_list[len(tpp_list) // 2]], lw=1.5,
                   label=rf"$p_\mathrm{{train}} = {a:g}$") for a in arm_list]
    leg = aT.legend(handles=_key, fontsize=7, loc="lower right", handlelength=1.4,
                    labelspacing=0.25, borderpad=0.35, handletextpad=0.5,
                    frameon=True, framealpha=0.92, edgecolor="0.75", fancybox=False)
    leg.get_frame().set_linewidth(0.5)

    fig.tight_layout(pad=0.3, h_pad=0.6)
    _save = None if SAVE_PATH is None else SAVE_PATH.format(arm=arm)
    if _save:
        # No bbox_inches="tight": tight_layout has set the margins, and "tight" would hand back a figure
        # wider than the column, which LaTeX then rescales.
        fig.savefig(_save, dpi=600)
        print(f"wrote {_save}")

    # The endpoints, for the caption: where each arm's temperature and residual land at the heaviest fault.
    P_HEAVY = max(q for key in _SHOWN for q in cell_grid(key))
    if NORMALIZED:
        print(f"fault axis in units of p_train = {P_REF:g}  (x = 1 is this arm's own training fault; "
              f"the clean arm trains at p_train = 0 and so has no scale of its own)")
        print(f"at p = {P_HEAVY:g}  (x = {P_HEAVY / P_REF:g}):")
    else:
        print(f"fault axis is the RAW p_eval; this arm trained at p_train = {arm:g}, so its own "
              f"condition sits at x = {arm:g} rather than at x = 1")
        print(f"at p = {P_HEAVY:g}:")
    # Medians over each cell's seed replicates, as the panels draw -- with the seed IQR on the two
    # numbers the caption quotes, so the figure and the text cannot disagree about the spread.
    hdr = (f"{'arm':>16}{'tok/param':>11}{'seeds':>6}{'T*':>7}{'resid (nats)':>14}"
           f"{r'$\eta/N$ (IQR)':>26}{'N_eff':>11}{'(clean law)':>13}{'explained':>11}")
    print(hdr); print("-" * len(hdr))
    for (arm, tpp) in _SHOWN:
        if P_HEAVY not in cell_grid((arm, tpp)):
            continue
        r0c = CELL[(arm, tpp)][0]
        _N = law_N(r0c)
        label = "clean" if arm == 0 else f"p_train={arm:g}"
        T, *_ = seed_band((arm, tpp), P_HEAVY, lambda j, w: FIT[(j, w)]["T"])
        res, *_ = seed_band((arm, tpp), P_HEAVY, lambda j, w: FIT[(j, w)]["residual_mean"])
        fr, *_ = seed_band((arm, tpp), P_HEAVY, lambda j, w: FIT[(j, w)]["frac"])
        own, olo, ohi, n = seed_band(
            (arm, tpp), P_HEAVY,
            lambda j, w: float(capacity_retained(FIT[(j, w)]["residual_mean"], _N, LAWS[arm])))
        cln, *_ = seed_band(                                  # same residual, priced on the reference law
            (arm, tpp), P_HEAVY,
            lambda j, w: float(capacity_retained(FIT[(j, w)]["residual_mean"], _N, LAW)))
        print(f"{label:>16}{tpp:>11g}{n:>6}{T:>7.3f}{res:>14.4f}"
              f"{f'{own:.4f} [{olo:.4f}, {ohi:.4f}]':>26}{own * _N:>11.3g}{cln:>13.4f}{fr:>11.1%}")
    return fig


for _arm in FAULT_ARMS:
    draw_arm(_arm)



p_train = 0.02  (vs the clean arm)
fault axis in units of p_train = 0.02  (x = 1 is this arm's own training fault; the clean arm trains at p_train = 0 and so has no scale of its own)
at p = 0.2  (x = 10):
             arm  tok/param seeds     T*  resid (nats)            $\eta/N$ (IQR)      N_eff  (clean law)  explained
-------------------------------------------------------------------------------------------------------------------
           clean          5     8  1.792        0.7442   0.0643 [0.0620, 0.0671]   3.18e+06       0.0643      61.9%
           clean         10     8  2.020        0.8090   0.0535 [0.0514, 0.0554]   2.65e+06       0.0535      68.0%
           clean         20     8  2.133        0.9335   0.0382 [0.0348, 0.0417]   1.89e+06       0.0382      68.4%
           clean         40     8  2.214        1.0408   0.0291 [0.0279, 0.0307]   1.44e+06       0.0291      68.5%
           clean         80     8  2.269        1.1131   0.0244 [0.0209, 0.0260]   1.21e+06       

In [14]:
# ---- paper figure: the temperature in the logits, 2x2 ------------------------------------
# Rows are training duration, columns the training arm. Each point is one vocabulary token in
# one context: x its clean logit z_0, y its faulted log(p_bar), both centred within the
# context, since the softmax is shift-invariant and only the spread matters. A point ON the
# temperature line is pure z -> z/T*; scatter off it is the residual the KL decomposition
# counts. The same T* as the figure above, so picture and metric cannot disagree.
SAVE_PATH_SCATTER = None      # e.g. "/home/trevor/figures/logit_scatter.pdf"; None -> just draw it

TPP_ROWS = [80, 640]          # <-- SET: the two durations, top row first. Must match the
                              # digest's cloud pass, or dg.cloud raises. Tokens per TOTAL
                              # parameter: this array's ladder is [5 ... 640].
ARMS_SHOW = [0.0, 0.04]       # <-- SET: the two COLUMNS. The grid is 2x2 and the array has six
                              # arms, so the pair is named rather than taken from arm_list.
P_SHOW = 0.04                 # <-- SET: eval fault; snapped to the nearest one present in ALL panels
N_SHOW = 30_000               # points drawn per panel (the digest stores ~250k; see the rasterize note)

_rng = np.random.default_rng(0)

# The clouds come from the digest, already centred and restricted to LIVE tokens. The
# never-predicted ones carry a near-constant very-negative logit the fault cannot move, so
# they would pile up on the diagonal and hide the cloud that matters; KL ignores them anyway.
# The threshold was fixed when the digest was built; change it there, not here.

# The four runs, and the fault they can all be shown at. p_train is prepended to a faulted
# job's grid as its own baseline, so the arms do NOT share every fault, and drawing the
# columns at different eval conditions is the one thing this comparison cannot survive.
# plm.select_panels is the SAME rule the digest's cloud pass used -- picking the panels twice
# by two rules that drift means the figure asks for a cloud the artifact does not carry.
_sel, P_DRAW = plm.select_panels(dg, TPP_ROWS, P_SHOW, arms=ARMS_SHOW)
_by_job = {r.job: r for r in runs}
panel = {(tpp, arm): _by_job[job]
         for (job, _p), (tpp, arm) in zip(_sel, [(t, a) for t in TPP_ROWS for a in ARMS_SHOW])}
if P_DRAW != P_SHOW:
    _common = sorted(set.intersection(*(set(r.p_eval) for r in panel.values())))
    print(f"p_eval {P_SHOW:g} is not in every panel's grid (it is a faulted-arm baseline); drawing all "
          f"four at {P_DRAW:g}, the nearest fault common to them\n  common grid: "
          f"{', '.join(f'{p:g}' for p in _common)}")

# One mid ramp step per arm rather than the per-duration colour of the figure above: at the alpha a
# 30k-point cloud needs, the light steps disappear, and the row label already carries the duration.
SCATTER_COL = {min(ARMS_SHOW): "#256abf", max(ARMS_SHOW): "#c44a1e"}

def _col_title(arm):
    """Name a column by its training fault. The faulted arm is drawn at (near enough) the fault it
    trained at, so it is labelled p_train = p_eval rather than repeating a number the x axis already
    fixes. That equality is CHECKED, not assumed: move P_SHOW away -- to the heaviest fault, say -- and
    the label falls back to the explicit rate instead of asserting something that stopped being true."""
    if arm > 0 and abs(np.log(arm / P_DRAW)) < 0.2:          # within ~20%, i.e. the same condition
        return r"$p_\mathrm{train} = p_\mathrm{eval}$"
    return rf"$p_\mathrm{{train}} = {arm:g}$"
INK = "#0b0b0b"

def _cloud(r):
    """The stored centred (z0, log p_bar) cloud for this panel, thinned to N_SHOW for drawing.

    dg.cloud raises -- naming what IS stored -- if this (job, fault) was not among the selections the
    digest was built for, rather than quietly drawing a different panel. Re-run the cloud pass of
    process_logit_marginals.py to add one; it reads a single point file, so it takes seconds."""
    c = dg.cloud(r.job, P_DRAW)
    z0c, zfc = c["z0"], c["z_fault"]
    idx = _rng.choice(z0c.size, min(N_SHOW, z0c.size), replace=False)
    return z0c[idx], zfc[idx], int(c["n_all"]), int(c["n_live"])

fig, axs = plt.subplots(2, 2, figsize=(3.4, 3.4), sharex=True, sharey=True)
_drawn = {}
for i, tpp in enumerate(TPP_ROWS):
    for j, arm in enumerate(ARMS_SHOW):
        r = panel[(tpp, arm)]
        ax = axs[i, j]
        x, y, n_all, n_live = _cloud(r)
        _drawn[(tpp, arm)] = (n_all, n_live)
        # rasterized: 30k vector points x 4 panels would be a ~10 MB PDF that a viewer chokes on. The
        # axes, lines and text stay vector, so only the cloud is a bitmap.
        ax.scatter(x, y, s=1.0, alpha=0.05, color=SCATTER_COL[arm], edgecolor="none",
                   rasterized=True, zorder=2)
        T = FIT[(r.job, P_DRAW)]["T"]
        lim = np.array([-1e3, 1e3])                      # drawn beyond the view; clipped to the axes
        ax.plot(lim, lim, color="0.6", lw=0.7, ls=":", zorder=3)          # y = x: the fault did nothing
        ax.plot(lim, lim / T, color=INK, lw=1.0, zorder=4)                # slope 1/T*: pure temperature
        if i == 0:
            ax.set_title(_col_title(arm), fontsize=8, pad=3)
        if j == len(ARMS_SHOW) - 1:
            ax.yaxis.set_label_position("right")
            # Both arms in a row share the duration, so either run gives the row's token count.
            _Drow = panel[(tpp, ARMS_SHOW[0])].n_train_tokens
            ax.set_ylabel(rf"$D$={_Drow / 1e9:.2g}B", fontsize=8, rotation=270, labelpad=11)
        ax.tick_params(labelsize=7.5, width=0.7, length=2.5, colors=INK_SOFT)
        ax.grid(True, which="major", color="0.92", lw=0.4, zorder=0)
        ax.set_axisbelow(True)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
        for side in ("left", "bottom"):
            ax.spines[side].set_linewidth(0.7)

# Shared limits from the pooled percentiles, not the extremes: a handful of far-out tokens would
# otherwise set the range and shrink the body of every cloud to a blob.
_all = np.concatenate([_cloud(r)[0] for r in panel.values()])
_hi = float(np.percentile(np.abs(_all), 99.5))
axs[0, 0].set_xlim(-_hi, _hi)
axs[0, 0].set_ylim(-_hi, _hi)
for ax in axs.ravel():
    ax.set_aspect("equal", adjustable="box")

fig.supxlabel(r"Clean logit  ($\propto \log P$)", fontsize=8, color=INK_SOFT, y=0.02)
fig.supylabel(r"Faulted logit  ($\propto \log Q$)", fontsize=8, color=INK_SOFT, x=0.015)
fig.tight_layout(pad=0.3, w_pad=0.4, h_pad=0.4, rect=(0.03, 0.04, 1, 1))
if SAVE_PATH_SCATTER:
    fig.savefig(SAVE_PATH_SCATTER, dpi=600)
    print(f"wrote {SAVE_PATH_SCATTER}")

print(f"\nlogit scatter at p_eval = {P_DRAW:g}  ({N_SHOW:,} of the points drawn per panel)")
hdr = (f"{'D (tokens)':>12}{'tok/param':>11}{'arm':>16}{'T*':>7}{'1/T*':>7}"
       f"{'live tokens':>13}{'resid (nats)':>14}")
print(hdr); print("-" * len(hdr))
for tpp in TPP_ROWS:
    for arm in ARMS_SHOW:
        r = panel[(tpp, arm)]
        d = FIT[(r.job, P_DRAW)]
        label = "clean" if arm == 0 else f"p_train={arm:g}"
        print(f"{r.n_train_tokens:>12.3g}{tpp:>11g}{label:>16}{d['T']:>7.3f}{1 / d['T']:>7.3f}"
              f"{_drawn[(tpp, arm)][1]:>13,}{d['residual_mean']:>14.4f}")


p_eval 0.04 is not in every panel's grid (it is a faulted-arm baseline); drawing all four at 0.0390993, the nearest fault common to them
  common grid: 0.002, 0.00212005, 0.00224731, 0.00238221, 0.0025252, 0.00267678, 0.00283745, 0.00300777, 0.00318832, 0.0033797, 0.00358257, 0.00379762, 0.00402557, 0.00426721, 0.00452335, 0.00479487, 0.00508269, 0.00538778, 0.00571118, 0.006054, 0.0064174, 0.00680261, 0.00721094, 0.00764379, 0.00810261, 0.00858898, 0.00910454, 0.00965104, 0.0102304, 0.0108444, 0.0114954, 0.0121854, 0.0129168, 0.0136922, 0.0145141, 0.0153853, 0.0163088, 0.0172878, 0.0183255, 0.0194255, 0.0205915, 0.0218275, 0.0231378, 0.0245266, 0.0259988, 0.0275594, 0.0292137, 0.0309673, 0.0328261, 0.0347966, 0.0368853, 0.0390993, 0.0414463, 0.0439341, 0.0465713, 0.0493668, 0.0523301, 0.0554713, 0.058801, 0.0623305, 0.066072, 0.070038, 0.0742421, 0.0786985, 0.0834225, 0.08843, 0.0937381, 0.0993648, 0.105329, 0.111652, 0.118354, 0.125458, 0.132989, 0.140971, 0.149433, 0.158403, 0.16791